# Explorador de Momento Óptimo con Alertas + Share of Wallet
> **Basado en la Sección 7 de `smart_demand_signals.ipynb`**

Este notebook extiende el explorador interactivo con dos nuevas capas de inteligencia:

1. **Alertas de Intervalo** — el sistema avisa proactivamente cuando:
   - ⚠️ El cliente **entra en su ventana de compra** (alerta anticipación al inicio del intervalo).
   - 🔴 La ventana **cierra sin que el cliente haya comprado** (alerta de cierre al final del intervalo).

2. **Share of Wallet y Velocidad del Share** — detecta el *momentum*:
   - Barras verdes = el cliente nos está comprando más que antes → fidelizar y hacer cross-sell.
   - Barras rojas = el cliente nos está comprando menos → riesgo de fuga a la competencia.


## 0. Setup e importaciones

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 20)

REFERENCE_DATE = pd.Timestamp('2025-12-30')
CUTOFF_12M     = REFERENCE_DATE - pd.DateOffset(months=12)

print(f'✅ Setup completado — Fecha de referencia: {REFERENCE_DATE.date()}')


## 1. Carga de datos

In [ ]:
comm = pd.read_csv('data/master_commodities.csv', parse_dates=['Fecha'])
tech = pd.read_csv('data/master_technicals.csv',  parse_dates=['Fecha'])

print(f'Commodities: {comm.shape[0]:,} filas, {comm.shape[1]} columnas')
print(f'Técnicos:    {tech.shape[0]:,} filas, {tech.shape[1]} columnas')
print(f'Rango fechas: {min(comm.Fecha.min(), tech.Fecha.min()).date()} → '
      f'{max(comm.Fecha.max(), tech.Fecha.max()).date()}')
print(f'Clientes únicos (commodities): {comm.Id_Cliente.nunique():,}')


## 2. Preprocesamiento

In [ ]:
# ── Baseline: excluir devoluciones y campañas ────────────────────────────────
comm_base = comm[(comm.es_devolucion == 0) & (comm.en_campana == 0)].copy()
tech_base = tech[(tech.es_devolucion == 0) & (tech.en_campana == 0)].copy()

# ── Agregación mensual (commodities) — necesaria para Share of Wallet ─────────
comm_base['year_month'] = comm_base.Fecha.dt.to_period('M')
monthly = comm_base.groupby(['Id_Cliente', 'Familia_Potencial', 'year_month']).agg(
    euros_venuts  = ('Valores_H',          'sum'),
    num_pedidos   = ('Num.Fact',           'nunique'),
    potencial_eur = ('Potencial_EUR_anual', 'first'),
).reset_index()
monthly['year_month_dt'] = monthly['year_month'].dt.to_timestamp()

# ── Gaps entre pedidos: Commodities (valid_gaps) ──────────────────────────────
pedidos_comm = (
    comm_base
    .groupby(['Id_Cliente', 'Familia_Potencial', 'Num.Fact'], sort=False)['Fecha']
    .min().reset_index()
    .sort_values(['Id_Cliente', 'Familia_Potencial', 'Fecha'])
)
pedidos_comm['gap_dies'] = (
    pedidos_comm.groupby(['Id_Cliente', 'Familia_Potencial'])['Fecha']
    .diff().dt.days
)
valid_gaps = pedidos_comm[pedidos_comm['gap_dies'] > 0]

# ── Pedidos técnicos ──────────────────────────────────────────────────────────
tech_pedidos = (
    tech_base
    .groupby(['Id_Cliente', 'Num.Fact'], sort=False)
    .agg(data=('Fecha', 'min'), euros=('Valores_H', 'sum'))
    .reset_index()
    .sort_values(['Id_Cliente', 'data'])
)
tech_pedidos['gap_dies'] = (
    tech_pedidos.groupby('Id_Cliente')['data'].diff().dt.days
)

# ── Ciclos poblacionales (para clientes nuevos sin historial propio) ──────────
pop_comm = (
    valid_gaps.groupby('Familia_Potencial')['gap_dies']
    .agg(pop_cicle_mig='mean', pop_cicle_std='std')
    .reset_index()
)
bio_g = tech_pedidos[tech_pedidos['gap_dies'] > 0]['gap_dies']
pop_bio = pd.DataFrame([{
    'Familia_Potencial': 'Biomateriales',
    'pop_cicle_mig': bio_g.mean() if len(bio_g) > 0 else 60.0,
    'pop_cicle_std': bio_g.std()  if len(bio_g) > 1 else 20.0,
}])
pop_cycle = pd.concat([pop_comm, pop_bio], ignore_index=True)

print('✅ Preprocesamiento completado')
print(f'   Pares (cliente, familia): {monthly.groupby(["Id_Cliente","Familia_Potencial"]).ngroups:,}')
print(f'   Gaps válidos commodities: {len(valid_gaps):,}')
print()
print('Ciclos poblacionales:')
display(pop_cycle.round(1))


## 3. Construcción de la timeline unificada

In [ ]:
# Commodities: un pedido = una fila
comm_tl = (
    comm_base
    .groupby(['Id_Cliente', 'Familia_Potencial', 'Num.Fact'])
    .agg(Fecha=('Fecha', 'min'), euros=('Valores_H', 'sum'))
    .reset_index()
    [['Id_Cliente', 'Familia_Potencial', 'Fecha', 'euros']]
)

# Técnicos: un pedido = una fila
tech_tl = (
    tech_base
    .groupby(['Id_Cliente', 'Num.Fact'])
    .agg(Fecha=('Fecha', 'min'), euros=('Valores_H', 'sum'))
    .reset_index()
    .assign(Familia_Potencial='Biomateriales')
    [['Id_Cliente', 'Familia_Potencial', 'Fecha', 'euros']]
)

# Unión y normalización a día 0 = primera compra
timeline = (
    pd.concat([comm_tl, tech_tl], ignore_index=True)
    .sort_values(['Id_Cliente', 'Familia_Potencial', 'Fecha'])
    .reset_index(drop=True)
)
timeline['primer_pedido'] = (
    timeline.groupby(['Id_Cliente', 'Familia_Potencial'])['Fecha'].transform('min')
)
timeline['dies_des_del_primer'] = (
    (timeline['Fecha'] - timeline['primer_pedido']).dt.days
)
timeline['n_pedidos_total'] = (
    timeline.groupby(['Id_Cliente', 'Familia_Potencial'])['Fecha'].transform('count')
)

print(f'✅ Timeline: {len(timeline):,} pedidos  |  '
      f'{timeline.Id_Cliente.nunique():,} clientes  |  '
      f'{timeline.Familia_Potencial.nunique()} familias')


---

## 4. Explorador Interactivo con Alertas de Intervalo

### Leyenda del gráfico
- 🔵 **Barras azules** = cliente establecido (ciclo personal EWM)
- 🟠 **Barras naranjas** = cliente nuevo (ciclo poblacional de la familia)
- 🟢 **Zona verde** = ventana de compra esperada (ciclo ±0.5σ)
- 🔴 **Zona roja** = zona de riesgo (retraso >0.5σ)
- ⬛ **Línea negra vertical** = hoy simulado

### Nuevas alertas
| Alerta | Cuándo se activa | Qué hacer |
|--------|-----------------|-----------|
| ⚠️ **ANTICIPACIÓN** | Hoy ha entrado en la ventana de compra y el cliente NO ha comprado aún | Contactar ahora, antes de que compre a la competencia |
| 🔴 **CIERRE** | La ventana ha cerrado y el cliente NO compró en todo el intervalo | ¡Llamada urgente! Alta probabilidad de pedido a la competencia |


In [ ]:
FAMILIES = ['Anestesia', 'Bioseguridad', 'Biomateriales']

# ── Helpers ───────────────────────────────────────────────────────────────────

def get_clients_for_familia(familia, min_n=1):
    mask = (timeline['Familia_Potencial'] == familia) & (timeline['n_pedidos_total'] >= min_n)
    return sorted(timeline.loc[mask, 'Id_Cliente'].unique().tolist())


def _get_client_timeline(client_id, familia):
    mask = (timeline['Id_Cliente'] == client_id) & (timeline['Familia_Potencial'] == familia)
    return timeline.loc[mask].sort_values('dies_des_del_primer').copy()


def _ewm_cycle(gaps_arr, half_life):
    n = len(gaps_arr)
    lam = np.log(2) / max(half_life, 0.1)
    weights = np.exp(lam * np.arange(n))
    weights /= weights.sum()
    cicle = float(np.dot(weights, gaps_arr))
    if n > 1:
        variance = float(np.dot(weights, (gaps_arr - cicle) ** 2))
        std = float(np.sqrt(variance))
    else:
        std = cicle * 0.30
    if np.isnan(std) or std <= 0:
        std = cicle * 0.30
    return cicle, std, weights


def _get_cycle(known_df, familia, es_nou, half_life):
    gaps = known_df['dies_des_del_primer'].diff().dropna()
    gaps = gaps[gaps > 0]
    if not es_nou and len(gaps) >= 1:
        cicle, cicle_std, weights = _ewm_cycle(gaps.values.astype(float), half_life)
        return cicle, cicle_std, weights, 'personal (EWM)', '#1565C0', '#1565C0'
    else:
        pop = pop_cycle[pop_cycle['Familia_Potencial'] == familia]
        cicle     = float(pop.iloc[0]['pop_cicle_mig']) if len(pop) > 0 else 45.0
        cicle_std = float(pop.iloc[0]['pop_cicle_std']) if len(pop) > 0 else 15.0
        if np.isnan(cicle_std) or cicle_std <= 0:
            cicle_std = cicle * 0.30
        return cicle, cicle_std, np.array([]), 'poblacional', '#E65100', '#E65100'


# ── Widgets ───────────────────────────────────────────────────────────────────

familia_w = widgets.Dropdown(
    options=FAMILIES, value='Anestesia',
    description='Familia:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='240px')
)
client_w = widgets.Dropdown(
    options=get_clients_for_familia('Anestesia'),
    description='Cliente ID:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='260px')
)
min_pedidos_w = widgets.IntSlider(
    value=3, min=1, max=10, step=1,
    description='Min pedidos (establecido):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='440px')
)
dia_avui_w = widgets.IntSlider(
    value=1000, min=1, max=2000, step=5,
    description='Hoy simulado (días):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px')
)
half_life_w = widgets.FloatSlider(
    value=3.0, min=0.5, max=10.0, step=0.5,
    description='Vida media (gaps):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='440px'),
    readout_format='.1f'
)
out = widgets.Output()


def _update_slider_for_client(client_id, familia):
    cdata = _get_client_timeline(client_id, familia)
    if len(cdata) == 0:
        return
    primer_date    = cdata['primer_pedido'].iloc[0]
    dies_avui_real = (REFERENCE_DATE - primer_date).days
    first_gap = int(cdata['dies_des_del_primer'].iloc[1]) if len(cdata) > 1 else 10
    dia_avui_w.min   = max(first_gap, 1)
    dia_avui_w.max   = dies_avui_real
    initial = max(int(dies_avui_real * 0.60), dia_avui_w.min)
    dia_avui_w.value = initial


def on_familia_change(change):
    new_clients = get_clients_for_familia(change['new'])
    client_w.options = new_clients
    if new_clients:
        client_w.value = new_clients[0]
        _update_slider_for_client(new_clients[0], change['new'])


def on_client_change(change):
    _update_slider_for_client(change['new'], familia_w.value)


familia_w.observe(on_familia_change, names='value')
client_w.observe(on_client_change,   names='value')


# ── Función principal de plot ─────────────────────────────────────────────────

def plot_client_timeline(client_id, familia, min_pedidos_establert, dia_avui_sim, half_life):
    cdata = _get_client_timeline(client_id, familia)
    if len(cdata) == 0:
        print(f'Cliente {client_id} sin datos para {familia}')
        return

    primer_date      = cdata['primer_pedido'].iloc[0]
    known  = cdata[cdata['dies_des_del_primer'] <= dia_avui_sim]
    future = cdata[cdata['dies_des_del_primer'] >  dia_avui_sim]
    n_known = len(known)
    if n_known == 0:
        print('Desplaza el slider a la derecha: no hay ninguna compra conocida.')
        return

    dies_ultim_known = int(known['dies_des_del_primer'].max())
    es_nou = n_known < min_pedidos_establert

    cicle, cicle_std, norm_w, cicle_tipus, bar_color, cicle_color = _get_cycle(
        known, familia, es_nou, half_life
    )
    gap_positions = known['dies_des_del_primer'].values[1:]

    # Ventanas de predicción
    preds = []
    for offset in range(1, 12):
        nd = dies_ultim_known + offset * cicle
        if nd > dies_ultim_known + 5 * cicle:
            break
        preds.append({
            'day':       nd,
            'low':       nd - 0.5 * cicle_std,
            'high':      nd + 0.5 * cicle_std,
            'risk_high': nd + 1.5 * cicle_std,
        })

    # ── DETECCIÓN DE ALERTAS ──────────────────────────────────────────────────
    anticipation_alerts = []
    cierre_alerts       = []

    for p in preds:
        purchases_in_window = known[
            (known['dies_des_del_primer'] >= p['low']) &
            (known['dies_des_del_primer'] <= p['high'])
        ]
        has_purchase = len(purchases_in_window) > 0

        # Alerta anticipación: hoy está DENTRO de la ventana pero sin compra
        if p['low'] <= dia_avui_sim <= p['high'] and not has_purchase:
            anticipation_alerts.append(p)

        # Alerta cierre: hoy está EN LA ZONA DE RIESGO (pasada la ventana) y sin compra
        elif p['high'] < dia_avui_sim <= p['risk_high'] and not has_purchase:
            cierre_alerts.append(p)

    # ── Backtesting hit/miss ──────────────────────────────────────────────────
    hit_set = set()
    for _, row in future.iterrows():
        d = row['dies_des_del_primer']
        if any(p['low'] <= d <= p['high'] for p in preds):
            hit_set.add(d)

    n_future    = len(future)
    n_hits      = len(hit_set)
    hit_rate    = n_hits / n_future if n_future > 0 else None
    future_hit  = future[future['dies_des_del_primer'].isin(hit_set)]
    future_miss = future[~future['dies_des_del_primer'].isin(hit_set)]

    # ── PLOT ──────────────────────────────────────────────────────────────────
    fig, axes2 = plt.subplots(
        2, 1, figsize=(15, 6.5),
        gridspec_kw={'height_ratios': [5, 1.2], 'hspace': 0.06}
    )
    ax, ax_w = axes2
    ymax  = float(cdata['euros'].max()) if cdata['euros'].max() > 0 else 1.0
    bar_w = max(cicle * 0.05, 3)

    # Ventanas de predicción
    for p in preds:
        ax.axvspan(p['low'],  p['high'],      alpha=0.14, color='green', zorder=1)
        ax.axvspan(p['high'], p['risk_high'], alpha=0.08, color='red',   zorder=1)
        ax.axvline(p['day'], color=cicle_color, ls='--', lw=0.9, alpha=0.5, zorder=2)

    # ── Marcadores de alerta ──────────────────────────────────────────────────
    for i, p in enumerate(anticipation_alerts):
        ax.axvline(p['low'], color='#FF6F00', lw=3, zorder=8, alpha=0.9)
        y_ann = ymax * (0.82 - i * 0.12)
        ax.annotate(
            '\u26a0\ufe0f INICIO VENTANA\nCONTACTAR AHORA',
            xy=(p['low'], y_ann * 0.88),
            xytext=(p['low'] + cicle * 0.15, y_ann),
            fontsize=7.5, color='#BF360C', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#FF6F00', lw=1.5),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF3E0',
                      edgecolor='#FF6F00', lw=1.5),
            zorder=9
        )

    for i, p in enumerate(cierre_alerts):
        ax.axvline(p['high'], color='#B71C1C', lw=3, zorder=8, alpha=0.9)
        y_ann = ymax * (0.65 - i * 0.12)
        ax.annotate(
            '\U0001f534 FIN VENTANA\nSIN COMPRA — \u00a1URGENTE!',
            xy=(p['high'], y_ann * 0.88),
            xytext=(p['high'] + cicle * 0.15, y_ann),
            fontsize=7.5, color='#B71C1C', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#B71C1C', lw=1.5),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFEBEE',
                      edgecolor='#B71C1C', lw=1.5),
            zorder=9
        )

    # Barras historial con intensidad EWM
    known_rows = list(known.iterrows())
    for idx, (_, row) in enumerate(known_rows):
        d = row['dies_des_del_primer']
        if idx > 0 and len(norm_w) > 0 and (idx - 1) < len(norm_w):
            w_min, w_max = norm_w.min(), norm_w.max()
            alpha_bar = 0.28 + 0.67 * (norm_w[idx - 1] - w_min) / max(w_max - w_min, 1e-9)
        else:
            alpha_bar = 0.40
        ax.bar(d, row['euros'], width=bar_w, color=bar_color, alpha=alpha_bar, zorder=3)
        if row['euros'] > 0:
            ax.text(d, row['euros'] + ymax * 0.015, f'{row["euros"]:.0f}',
                    ha='center', va='bottom', fontsize=7.5, color='#222')

    if len(future_hit) > 0:
        ax.bar(future_hit['dies_des_del_primer'], future_hit['euros'],
               width=bar_w, color='#2E7D32', alpha=0.75, zorder=4)
    if len(future_miss) > 0:
        ax.bar(future_miss['dies_des_del_primer'], future_miss['euros'],
               width=bar_w, color='#90A4AE', alpha=0.55, zorder=3)
    for _, row in future.iterrows():
        if row['euros'] > 0:
            c = '#1B5E20' if row['dies_des_del_primer'] in hit_set else '#546E7A'
            ax.text(row['dies_des_del_primer'], row['euros'] + ymax * 0.015,
                    f'{row["euros"]:.0f}', ha='center', va='bottom',
                    fontsize=7.5, color=c)

    ax.axvline(dia_avui_sim, color='black', lw=2.4, zorder=6)

    estat   = 'NUEVO' if es_nou else 'ESTABLECIDO'
    hit_txt = (f'  Tasa acierto: {n_hits}/{n_future} = {hit_rate*100:.0f}%'
               if hit_rate is not None else '  Sin compras futuras')
    ax.set_title(
        f'Cliente {client_id}  |  {familia}  |  {estat}  |  '
        f'Ciclo {cicle_tipus}: {cicle:.0f} días (+/-{cicle_std:.0f}){hit_txt}\n'
        f'Intensidad barras = peso EWM  |  half_life={half_life:.1f} gaps',
        fontsize=10, pad=8
    )
    ax.set_ylabel('Importe (€)', fontsize=10)

    handles = [
        mpatches.Patch(color=bar_color, alpha=0.75,
                       label=f'Historial ({n_known} pedidos) — intensidad = peso EWM'),
        mpatches.Patch(color='green', alpha=0.30, label='Ventana esperada (±0.5σ)'),
        mpatches.Patch(color='red',   alpha=0.20, label='Zona de riesgo (>0.5σ retraso)'),
        plt.Line2D([0], [0], color='black', lw=2,   label=f'Hoy simulado (día {dia_avui_sim})'),
        plt.Line2D([0], [0], color='#FF6F00', lw=2.5, label='⚠️ Alerta Anticipación (inicio ventana)'),
        plt.Line2D([0], [0], color='#B71C1C', lw=2.5, label='🔴 Alerta Cierre (fin ventana sin compra)'),
    ]
    if len(future_hit)  > 0:
        handles.append(mpatches.Patch(color='#2E7D32', alpha=0.75, label=f'Acierto ({len(future_hit)})'))
    if len(future_miss) > 0:
        handles.append(mpatches.Patch(color='#90A4AE', alpha=0.55, label=f'No acierto ({len(future_miss)})'))
    ax.legend(handles=handles, loc='upper left', fontsize=8)
    ax.grid(axis='y', alpha=0.3)

    x_right = max(dia_avui_sim * 1.05,
                  dies_ultim_known + 3.5 * cicle,
                  cdata['dies_des_del_primer'].max() * 1.03)
    ax.set_xlim(-bar_w * 2, x_right)
    ax.set_ylim(0, ymax * 1.22)
    ax.set_xticklabels([])

    # Sub-plot pesos EWM
    if len(norm_w) > 0:
        cmap_vals   = 0.3 + 0.7 * norm_w / norm_w.max()
        bar_colors_w = plt.cm.Blues(cmap_vals)
        ax_w.bar(gap_positions, norm_w * 100, width=bar_w * 1.5,
                 color=bar_colors_w, alpha=0.9)
        for gp, gw in zip(gap_positions, norm_w):
            ax_w.text(gp, gw * 100 + norm_w.max() * 2, f'{gw*100:.1f}%',
                      ha='center', va='bottom', fontsize=7, color='#333')
        ax_w.axvline(dia_avui_sim, color='black', lw=1.5, alpha=0.4)
        ax_w.set_xlim(ax.get_xlim())
        ax_w.set_ylim(0, norm_w.max() * 140)
        ax_w.set_ylabel('Peso EWM\n(%)', fontsize=7.5)
        ax_w.grid(axis='y', alpha=0.2)
        ax_w.tick_params(labelsize=7)
    else:
        ax_w.axis('off')
    ax_w.set_xlabel('Días desde el primer pedido  (día 0 = primera compra)', fontsize=10)

    plt.tight_layout()
    plt.show()

    # ── Resumen texto ─────────────────────────────────────────────────────────
    print(f"{'─'*62}")
    print(f"  Backtesting EWM  |  Cliente {client_id}  |  {familia}")
    print(f"{'─'*62}")
    print(f"  Hoy simulado:    día {dia_avui_sim}  "
          f"({(primer_date + pd.Timedelta(days=dia_avui_sim)).strftime('%d/%m/%Y')})")
    print(f"  Historial:       {n_known} pedidos  (hasta día {dies_ultim_known})")
    if len(norm_w) > 0:
        print(f"  EWM half_life:   {half_life:.1f} gaps  "
              f"→ peso último gap: {norm_w[-1]*100:.1f}%  |  primer gap: {norm_w[0]*100:.1f}%")
    print(f"  Ciclo EWM:       {cicle:.0f} días ± {cicle_std:.0f}  ({cicle_tipus})")
    print(f"  Compras futuras: {n_future}")
    if hit_rate is not None:
        stars = '★' * min(int(hit_rate * 5), 5)
        print(f"  Aciertos:        {n_hits} / {n_future}  ({hit_rate*100:.0f}%)  {stars}")
    print(f"{'─'*62}")

    # ── Bloque de alertas ─────────────────────────────────────────────────────
    print()
    if anticipation_alerts or cierre_alerts:
        print(f"{'═'*62}")
        print(f"  🚨 ALERTAS ACTIVAS EN EL MOMENTO SIMULADO")
        print(f"{'═'*62}")
        for p in anticipation_alerts:
            d_ini  = (primer_date + pd.Timedelta(days=int(p['low']))).strftime('%d/%m/%Y')
            d_fin  = (primer_date + pd.Timedelta(days=int(p['high']))).strftime('%d/%m/%Y')
            print(f"  ⚠️  ALERTA ANTICIPACIÓN — INICIO DE VENTANA DE COMPRA")
            print(f"     El cliente HA ENTRADO en su ventana prevista ({d_ini} → {d_fin}).")
            print(f"     No se ha registrado ninguna compra todavía.")
            print(f"     ACCIÓN: Contactar AHORA para asegurar el pedido antes que la")
            print(f"             competencia. Es el momento óptimo de intervención.")
            print()
        for p in cierre_alerts:
            d_ini  = (primer_date + pd.Timedelta(days=int(p['low']))).strftime('%d/%m/%Y')
            d_fin  = (primer_date + pd.Timedelta(days=int(p['high']))).strftime('%d/%m/%Y')
            d_risk = (primer_date + pd.Timedelta(days=int(p['risk_high']))).strftime('%d/%m/%Y')
            print(f"  🔴 ALERTA CIERRE — FIN DE VENTANA SIN COMPRA")
            print(f"     La ventana prevista ({d_ini} → {d_fin}) ha CERRADO sin compra.")
            print(f"     Zona de riesgo activa hasta: {d_risk}")
            print(f"     ACCIÓN: ¡Llamada URGENTE! Alta probabilidad de que el cliente")
            print(f"             haya comprado (o esté a punto de comprar) a la competencia.")
            print()
        print(f"{'═'*62}")
    else:
        print(f"  ✅ Sin alertas de intervalo activas en el momento simulado.")
        print(f"{'═'*62}")


def update_chart(change=None):
    with out:
        clear_output(wait=True)
        if client_w.value is not None:
            plot_client_timeline(
                client_w.value, familia_w.value,
                min_pedidos_w.value, dia_avui_w.value,
                half_life_w.value
            )


familia_w.observe(update_chart,     names='value')
client_w.observe(update_chart,      names='value')
min_pedidos_w.observe(update_chart, names='value')
dia_avui_w.observe(update_chart,    names='value')
half_life_w.observe(update_chart,   names='value')

header = widgets.HTML(
    '<h3 style="margin:4px 0;color:#1565C0">Explorador + Alertas de Intervalo (EWM)</h3>'
    '<p style="color:#555;margin:2px 0">'
    'Ciclo con <b>decaimiento exponencial</b>: los gaps más recientes pesan más. '
    '<b>⚠️ Naranja</b> = alerta al inicio del intervalo de compra (anticipación). '
    '<b>🔴 Rojo</b> = alerta al final del intervalo si el cliente no ha comprado.</p>'
)
row1 = widgets.HBox([familia_w, client_w],
                    layout=widgets.Layout(gap='10px', align_items='center'))
row2 = widgets.HBox([min_pedidos_w, half_life_w],
                    layout=widgets.Layout(gap='10px', align_items='center'))
row3 = widgets.HBox([dia_avui_w])
display(widgets.VBox([header, row1, row2, row3, out]))

_update_slider_for_client(client_w.value, familia_w.value)
update_chart()


---

## 5. Share of Wallet y Velocidad del Share (Clientes Promiscuos)

### 1. ¿Cuál es la métrica clave? (El "Share of Wallet")
Para un cliente promiscuo, no solo nos importa cuánto nos compra en euros, sino *qué porcentaje de su capacidad total de compra (potencial) nos está destinando a nosotros*. A esto se le llama **Share of Wallet**.

- **Cálculo:** Se suman las ventas de los últimos 12 meses (`rolling_12m`) y se dividen entre el potencial anual del cliente (`Potencial_EUR_anual`).

### 2. ¿Cómo se detecta el momento óptimo? (La "Velocidad del Share")
El modelo no se fija solo en la "foto actual" (qué porcentaje de share tenemos hoy), sino en la *película* (hacia dónde va la tendencia). Para ello se calcula la **Velocidad del Share** (técnicamente, la primera derivada), que es simplemente la diferencia del Share entre este mes y el mes anterior.

Esto genera dos escenarios claros de actuación:

- **Velocidad Positiva (Barras Verdes en el gráfico):** El cliente está acelerando sus compras con nosotros y quitándoselas a la competencia.
  - **Acción de Negocio:** Es el momento de **fidelizar e intentar ventas cruzadas (cross-sell)**, ya que el cliente está receptivo a nuestra marca.

- **Velocidad Negativa (Barras Rojas en el gráfico):** Nuestro Share está cayendo. El cliente está empezando a desviar sus compras hacia un competidor.
  - **Acción de Negocio (Riesgo Inminente):** Si la velocidad de caída es drástica (umbral: caída > 5% mensual), salta una alerta. El comercial debe llamar inmediatamente, posiblemente con una **oferta agresiva** para frenar la fuga antes de que el competidor se haga con todo el negocio.

### En resumen para el equipo de ventas
El modelo de "clientes promiscuos" les dice a los comerciales que dejen de mirar solo la facturación absoluta. Les avisa de forma temprana (basándose en los cambios de tendencia o *momentum*) cuándo un cliente está a punto de irse con la competencia para que actúen con urgencia, o cuándo está en un ciclo de "enamoramiento" con Inibsa para que aprovechen y le vendan más cosas.

| Velocidad | Señal | Acción |
|-----------|-------|--------|
| > +5pp/mes (🟢) | Cliente acelerando con nosotros | Cross-sell, fidelizar |
| Entre -5 y +5pp/mes (🔵) | Share estable | Seguimiento estándar |
| < -5pp/mes (🔴) | Fuga hacia competidor | Llamada urgente con oferta agresiva |


In [ ]:
# ── Widgets Share of Wallet ───────────────────────────────────────────────────

SOW_FAMILIES = ['Anestesia', 'Bioseguridad']  # Solo familias con datos de potencial

def get_sow_clients(familia, min_meses=3):
    mask = (
        (monthly['Familia_Potencial'] == familia) &
        (monthly['potencial_eur'] > 0)
    )
    counts = monthly[mask].groupby('Id_Cliente')['year_month'].count()
    return sorted(counts[counts >= min_meses].index.tolist())


sow_familia_w = widgets.Dropdown(
    options=SOW_FAMILIES, value='Anestesia',
    description='Familia:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='240px')
)
sow_client_w = widgets.Dropdown(
    options=get_sow_clients('Anestesia'),
    description='Cliente ID:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='260px')
)
out_sow = widgets.Output()


def on_sow_familia_change(change):
    new_clients = get_sow_clients(change['new'])
    sow_client_w.options = new_clients
    if new_clients:
        sow_client_w.value = new_clients[0]


sow_familia_w.observe(on_sow_familia_change, names='value')


def plot_share_wallet(client_id, familia):
    mask = (monthly['Id_Cliente'] == client_id) & (monthly['Familia_Potencial'] == familia)
    cm   = monthly[mask].copy().sort_values('year_month_dt')

    if len(cm) < 3:
        print("Datos insuficientes (mínimo 3 meses de historial).")
        return

    potencial = cm['potencial_eur'].iloc[0]
    if potencial <= 0:
        print("Potencial del cliente = 0. No se puede calcular el Share of Wallet.")
        return

    # ── Reindexar a TODOS los meses del calendario ────────────────────────────
    # Sin este paso los meses sin compra no tienen fila: el rolling(12) los
    # ignora y las ventas de hace >12 meses nunca salen de la ventana,
    # dejando el share artificialmente pegado en 100% aunque el cliente
    # lleve años sin comprar.
    cm = cm.set_index('year_month_dt')
    all_months = pd.date_range(cm.index.min(), REFERENCE_DATE, freq='MS')
    cm = cm.reindex(all_months)
    cm['euros_venuts']  = cm['euros_venuts'].fillna(0)
    cm['potencial_eur'] = cm['potencial_eur'].ffill().bfill()

    # Cálculo de Share y Velocidad sobre la serie completa de meses
    cm['rolling_12m']    = cm['euros_venuts'].rolling(12, min_periods=1).sum()
    cm['share']          = (cm['rolling_12m'] / potencial).clip(0, 1)
    cm['share_velocity'] = cm['share'].diff()
    cm = cm.reset_index().rename(columns={'index': 'year_month_dt'})

    # ── Gráfico ────────────────────────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(13, 7),
        gridspec_kw={'height_ratios': [2.5, 1.5], 'hspace': 0.10}
    )

    # Top: Share of Wallet
    ax1.plot(cm['year_month_dt'], cm['share'], color='#1565C0',
             marker='o', lw=2.2, markersize=3, label='Share of Wallet (rolling 12m)')
    ax1.fill_between(cm['year_month_dt'], cm['share'], alpha=0.08, color='#1565C0')
    ax1.axhline(0.70, color='#2E7D32', ls='--', lw=1.2, alpha=0.7, label='70% (umbral fidelidad)')
    ax1.axhline(0.20, color='#C62828', ls='--', lw=1.2, alpha=0.7, label='20% (umbral riesgo)')
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax1.set_ylabel('Share of Wallet', fontsize=10)
    ax1.set_ylim(0, 1.1)
    ax1.set_title(
        f'Share of Wallet — Cliente {client_id} | {familia}\n'
        f'Potencial anual: {potencial:,.0f} €  |  '
        f'Capturado últimos 12m: {cm["rolling_12m"].iloc[-1]:,.0f} €  |  '
        f'Recuperable: {max(0, potencial - cm["rolling_12m"].iloc[-1]):,.0f} €',
        fontsize=11
    )
    ax1.legend(fontsize=9)
    ax1.set_xticklabels([])
    ax1.grid(axis='y', alpha=0.3)

    # Bottom: Velocidad del Share
    vel_pp = cm['share_velocity'].fillna(0) * 100  # en puntos porcentuales
    colors_vel = ['#2E7D32' if v >= 0 else '#C62828' for v in vel_pp]
    ax2.bar(cm['year_month_dt'], vel_pp, color=colors_vel, alpha=0.75, width=25)
    ax2.axhline(0,   color='gray',    lw=1)
    ax2.axhline(-5,  color='#C62828', lw=1.3, ls='--', label='Umbral riesgo (-5pp/mes)')
    ax2.axhline(5,   color='#2E7D32', lw=1.3, ls='--', label='Umbral oportunidad (+5pp/mes)')
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:+.1f}pp'))
    ax2.set_ylabel('Velocidad del Share\n(pp/mes)', fontsize=10)
    ax2.set_xlabel('Fecha', fontsize=10)
    ax2.legend(fontsize=8)
    ax2.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()

    # ── Resumen y alerta ──────────────────────────────────────────────────────
    last_vel    = float(cm['share_velocity'].iloc[-1]) if len(cm) > 1 else 0.0
    last_share  = float(cm['share'].iloc[-1])
    last_12m    = float(cm['rolling_12m'].iloc[-1])
    recuperable = max(0.0, potencial - last_12m)

    print(f"{'═'*62}")
    print(f"  Análisis Share of Wallet — Cliente {client_id} | {familia}")
    print(f"{'═'*62}")
    print(f"  Share actual (rolling 12m): {last_share:.1%}")
    print(f"  Velocidad actual:           {last_vel:+.1%}/mes")
    print(f"  Potencial anual:            {potencial:,.0f} €")
    print(f"  Euros capturados (12m):     {last_12m:,.0f} €")
    print(f"  Euros recuperables:         {recuperable:,.0f} €")
    print(f"{'─'*62}")

    if last_vel < -0.05:
        print(f"  🔴 ALERTA: RIESGO INMINENTE DE FUGA")
        print(f"     El Share está cayendo {last_vel:.1%} por mes.")
        print(f"     El cliente está desviando compras hacia la competencia.")
        print(f"     ACCIÓN: Llamar con OFERTA AGRESIVA para frenar la fuga.")
        print(f"             Potencial en riesgo: {recuperable:,.0f} € adicionales.")
    elif last_vel > 0.05:
        print(f"  🟢 OPORTUNIDAD: CLIENTE EN FASE DE FIDELIZACIÓN")
        print(f"     El Share está creciendo {last_vel:.1%} por mes.")
        print(f"     El cliente está receptivo a nuestra marca.")
        print(f"     ACCIÓN: Intentar CROSS-SELL — el cliente está en su mejor")
        print(f"             momento de receptividad. Recuperable: {recuperable:,.0f} €")
    elif last_share < 0.20:
        print(f"  🟡 ATENCIÓN: Share muy bajo ({last_share:.1%})")
        print(f"     La competencia domina a este cliente.")
        print(f"     ACCIÓN: Revisar estrategia de cuenta. Oportunidad de {recuperable:,.0f} €")
    else:
        print(f"  🔵 SEGUIMIENTO ESTÁNDAR")
        print(f"     Share estable ({last_share:.1%}). Mantener contacto habitual.")
    print(f"{'═'*62}")


def update_sow(change=None):
    with out_sow:
        clear_output(wait=True)
        if sow_client_w.value is not None:
            plot_share_wallet(sow_client_w.value, sow_familia_w.value)


sow_familia_w.observe(update_sow, names='value')
sow_client_w.observe(update_sow,  names='value')

sow_header = widgets.HTML(
    '<h3 style="margin:4px 0;color:#E65100">Share of Wallet — Explorador por Cliente</h3>'
    '<p style="color:#555;margin:2px 0">'
    '<b>🟢 Verde</b> = velocidad positiva (ganar terreno). '
    '<b>🔴 Rojo</b> = velocidad negativa (perder terreno). '
    'Umbral de alerta: ±5 puntos porcentuales por mes.</p>'
)
sow_row1 = widgets.HBox([sow_familia_w, sow_client_w],
                         layout=widgets.Layout(gap='10px', align_items='center'))
display(widgets.VBox([sow_header, sow_row1, out_sow]))

update_sow()
